In [1]:
import cmdstanpy

# Point CmdStanPy at your existing local CmdStan build.
# Alternative: set a CMDSTAN environment variable and delete this line,
# to keep the machine-specific path out of committed code.
cmdstanpy.set_cmdstan_path("/Users/marobinette/cmdstan")

print("cmdstanpy", cmdstanpy.__version__)
print("cmdstan", ".".join(map(str, cmdstanpy.cmdstan_version())))

/opt/anaconda3/envs/stan/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cmdstanpy 1.3.0
cmdstan 2.40


In [3]:
stan_code = """
data {
  int<lower=1> N;                        // measurements per simulated dataset
}
generated quantities {
  real mu = normal_rng(160, 10);         // prior on mean height
  real<lower=0> sigma2 = uniform_rng(0, 5000);   // uniform prior on the variance
  array[N] real y_sim;
  for (i in 1:N)
    y_sim[i] = normal_rng(mu, sqrt(sigma2));      // likelihood takes the sd
}
"""

In [6]:
from cmdstanpy import CmdStanModel
from pathlib import Path

stan_file = Path("height_prior_pred.stan")
stan_file.write_text(stan_code)

m = CmdStanModel(stan_file=str(stan_file.resolve()))
fit = m.sample(data={"N": 20}, fixed_param=True,
               iter_sampling=50, chains=1, seed=1)

mu     = fit.stan_variable("mu")      # (50,)
sigma2 = fit.stan_variable("sigma2")  # (50,)
y      = fit.stan_variable("y_sim")   # (50, 20)  -> 1000 heights

print(y.shape)
print("fraction < 0:   ", (y < 0).mean())
print("fraction > 300: ", (y > 300).mean())

15:34:48 - cmdstanpy - INFO - compiling stan file /Users/marobinette/bayes/height_prior_pred.stan to exe file /Users/marobinette/bayes/height_prior_pred
15:34:56 - cmdstanpy - INFO - compiled model executable: /Users/marobinette/bayes/height_prior_pred
15:34:56 - cmdstanpy - INFO - CmdStan start processing
chain 1: 100%|██████████| 1050/1050 [00:00<00:00, 32874.16it/s, (Sampling completed)]


15:34:57 - cmdstanpy - INFO - CmdStan done processing.



(50, 20)
fraction < 0:    0.004
fraction > 300:  0.008


In [25]:
import numpy as np

mu_out, sigma2_out, y_out = [], [], []

# for iteration in range(50):                      # <-- iter_sampling=50, the INVISIBLE outer loop
    # ---- this indented body is your `generated quantities` block ----
mu     = np.random.normal(160, 10)           # one scalar, drawn once
sigma2 = np.random.uniform(0, 5000)          # one scalar, drawn once
y_sim  = np.empty(20)                         # N = 20
for i in range(20):                           # the VISIBLE inner loop
    y_sim[i] = np.random.normal(mu, np.sqrt(sigma2))
# ---- end of block ----
mu_out.append(mu)                             # Stan records every top-level
sigma2_out.append(sigma2)                     # variable in the block, once
y_out.append(y_sim)                           # per iteration

mu_out    = np.array(mu_out)      # (50,)
sigma2_out= np.array(sigma2_out)  # (50,)
y_out     = np.array(y_out)       # (50, 20)
print(y_sim)

[176.41944288 135.85417686 145.17266177 113.0319431  126.84663481
 217.82866039 -18.06587636 170.98301947 117.68558391 197.25608705
 126.8910349  130.88424772 153.81210768  69.42244915  78.93247569
 141.51855928 136.72273299 138.60857688 167.66429811 111.7809035 ]
